(course-core-covalent-connectivity)=

# Module 13: Topological Analysis

**Welcome back, Apprentice Master.** In [Module 12: Building, Repairing and Auditing Systems](../Common_Core/12_Building_Repairing_and_Auditing_Systems.ipynb), you explored system quality auditing and structural repair with `msm.build`. Now we turn to non-spatial features: **Topological Analysis** using **`msm.chemical_states`** for covalent graphs and **`msm.topology`** for sequence comparisons.

In computational biophysics, a molecular system contains properties that are entirely independent of 3D spatial coordinates—such as covalent graph connectivity, connected molecular components, pattern-matched covalent walks, and primary amino acid sequence ordering. Coordinate structures and chemical states are independent axes. A state defines its own covalent graph; stable inventory and amino-acid sequence do not change when another stored chemical state is selected.

These two function modules provide complementary tools for non-spatial properties:
- **`msm.chemical_states.get_covalent_blocks()`**: Partitions connected covalent components and analyzes sub-blocks under hypothetical bond cuts.
- **`msm.chemical_states.get_covalent_paths()`**: Measures shortest topological bond paths between specified atom pairs.
- **`get_sequence_alignment()`**: Aligns residue sequences across different molecular systems.
- **`get_sequence_identity()`**: Computes percentage sequence identity between matching chains or systems.

(course-core-covalent-connectivity-learning-outcomes)=
```{admonition} Learning Outcomes
:class: dropdown learning-outcomes

By the end of this module, you will be able to:
- Partition connected covalent components using `msm.chemical_states.get_covalent_blocks()`.
- Simulate bond cuts to analyze rotatable groups using the `remove_bonds` parameter.
- Measure topological covalent paths between atom pairs with `msm.chemical_states.get_covalent_paths()`.
- Perform pairwise sequence alignment using `msm.topology.get_sequence_alignment()`.
- Calculate sequence identity percentages using `msm.topology.get_sequence_identity()`.
```

For explicitly selected bond indices, `msm.chemical_states.get_rigid_fragments()`
returns packed fragment memberships and an aligned fragment graph. It requires
complete declared connectivity and rejects ring and dative cuts. It does not
choose chemically rotatable bonds. See {ref}`Tutorial_Rigid_Fragments`.
:::{hint}
:class: dropdown
**msm.chemical_states.get_rigid_fragments()**: Partitions complete covalent connectivity
at explicit source bond indices. See {func}`molsysmt.chemical_states.get_rigid_fragments`.
:::

For chemical torsion candidates, `msm.chemical_states.get_rotatable_bonds()` classifies
the full declared graph before filtering source bond indices. Its named
acyclic-single and conjugation-restricted criteria include aligned exclusion
evidence. They do not measure energy barriers or choose docking ROOT. Pass
explicitly chosen cuts to the separate fragment tool. See
{ref}`Tutorial_Rotatable_Bonds` for analytical and original-ligand examples.
:::{hint}
:class: dropdown
**msm.chemical_states.get_rotatable_bonds()**: Classifies source torsion candidates
with declared criteria and exclusions. See {func}`molsysmt.chemical_states.get_rotatable_bonds`.
:::


## 1. Covalent Blocks and Paths

Let's begin by importing MolSysMT and loading our T4 Lysozyme demonstration system.
The optional experimental source
[MolSysMT tools addon for MolSysViewer](../../user/tools/molsysviewer_tools.md)
offers a received `view.addons.molsysmt.topology` slice for the current system:
sequence queries. The chemical-graph tools now belong to `msm.chemical_states`;
the receiving Viewer migration is tracked in [MolSysViewer #269](https://github.com/uibcdf/molsysviewer/issues/269). These return native scientific values without changing molecular
bonds or creating scene objects. Chemistry prerequisites, optional engines
and scientific limits remain native. The source now receives contributing
sequence-group mapping and the native validation flag repaired under
[MolSysMT #439](https://github.com/uibcdf/molsysmt/issues/439).


In [1]:
import molsysmt as msm
from molsysmt import systems

# Load T4 Lysozyme as native MolSys
lysozyme = msm.convert(systems['T4 lysozyme L99A']['181l.bcif.gz'], to_form='molsysmt.MolSys')

A **covalent block** is a set of atoms connected continuously through covalent bonds. When called without parameters, `msm.chemical_states.get_covalent_blocks()` partitions the system into independent connected components:

In [2]:
# Partition system into connected covalent blocks
blocks = msm.chemical_states.get_covalent_blocks(lysozyme)
n_components = msm.get(lysozyme, n_components=True)

print(f"Covalent blocks count: {len(blocks)}")
print(f"System components count: {n_components}")

Covalent blocks count: 141
System components count: 141


You can also use `remove_bonds` to simulate hypothetical bond cuts. For example, cutting an N-CA backbone bond splits a connected protein component into two sub-blocks:

In [3]:
# Select the N and CA atom indices of group index 20
n_atom = msm.select(lysozyme, selection='group_index == 20 and atom_name == "N"')[0]
ca_atom = msm.select(lysozyme, selection='group_index == 20 and atom_name == "CA"')[0]

# Partition blocks after hypothetically cutting the N-CA bond
blocks_after_cut = msm.chemical_states.get_covalent_blocks(lysozyme, remove_bonds=[[n_atom, ca_atom]])

print(f"Blocks count before bond cut: {len(blocks)}")
print(f"Blocks count after N-CA cut : {len(blocks_after_cut)}")

Blocks count before bond cut: 141
Blocks count after N-CA cut : 142


Use `msm.chemical_states.get_covalent_paths()` to enumerate bonded walks matching an ordered pattern. Each entry in `path` supplies the candidates for one position. A one-position pattern `[[0, 1]]` therefore gives two one-atom walks, not the shortest path between atoms 0 and 1.

Returned atom indices always refer to the complete source system. Empty positions and disconnected candidates give an `int64` array with shape `(0, len(path))`; invalid selectors still raise an error.

In [4]:
# Enumerate the single-atom walks matching one candidate position
path_result = msm.chemical_states.get_covalent_paths(lysozyme, path=[[0, 1]])
print(f"One-position covalent walks:\n{path_result}")

One-position covalent walks:
[[0]
 [1]]


:::{hint}
:class: dropdown
**msm.chemical_states.get_covalent_blocks()**: Returns sets of atoms remaining connected when specified bonds are cut. See API doc: {func}`molsysmt.chemical_states.get_covalent_blocks`.
:::

## 2. Sequence Alignment and Identity

`msm.topology` provides sequence comparison utilities independently of the chemical graph. You can align residue sequences and calculate sequence identity percentages between different molecular systems.

Let's compare **T4 Lysozyme** (`181L`) against a related T4 Lysozyme variant structure (**PDB ID 5X33**):

In [5]:
# Load reference T4 Lysozyme variant
t4_variant = 'pdb_id:5X33'

# Compute sequence alignment
alignment = msm.topology.get_sequence_alignment('pdb_id:181L', reference_molecular_system=t4_variant)
print(f"Aligned T4 Lysozyme (181L) seq: {alignment[0][:60]}...")
print(f"Aligned T4 Variant  (5X33) seq: {alignment[1][:60]}...")

# Calculate sequence identity percentage
seq_id = msm.topology.get_sequence_identity('pdb_id:181L', reference_molecular_system=t4_variant)
print(f"Sequence Identity: {seq_id[0]:.2f}%")

Aligned T4 Lysozyme (181L) seq: ------------------------------------------------------------...
Aligned T4 Variant  (5X33) seq: SNTFIPLLAMILLSVSMVVGLPGNTFVVWSILKRMRKRSVTALMVLNLALADLAVLLTAP...


Sequence Identity: 98.15%


(course-core-covalent-connectivity-chemical-correspondence)=
## 3. Chemical correspondence

You can compare two declared chemical graphs without coordinates using `msm.chemical_states.get_atom_correspondences()`. The experimental profile retains complete covalent context and exact state restrictions. A benzene ring has twelve admitted rotations/reflections; a selected subset retains its unselected chemical environment. See {ref}`Tutorial_Get_atom_correspondences` for charge, H, stereo and search-limit requirements.

Partial `fixed_atom_correspondence` anchors constrain the chemical search, including unselected context atoms. Complete `candidate_atom_correspondences` restrict evaluation to validated caller-supplied maps. Keep `search_scope` with your result: a conditional minimum does not certify the unrestricted symmetry minimum. A raw baseline map does not impose either restriction.


In [6]:
import molsysmt as msm
correspondence_example = msm.chemical_states.get_atom_correspondences('smiles:c1ccccc1')
assert correspondence_example['atom_correspondences'].shape == (12, 6, 2)
print('Admitted complete-graph mappings:', len(correspondence_example['atom_correspondences']))
anchored_example = msm.chemical_states.get_atom_correspondences(
    "smiles:c1ccccc1", fixed_atom_correspondence=[[0, 0]], max_matches=2)
assert anchored_example["atom_correspondences"].shape == (2, 6, 2)
print("Declared search scope:", anchored_example["search_scope"])


Admitted complete-graph mappings: 12


Declared search scope: anchored_chemical_correspondences


:::{hint}
:class: dropdown
{func}`molsysmt.chemical_states.get_atom_correspondences` returns source-index bijections constrained by declared chemical states. Coordinates are unused. Methodological inspiration and executed software are retained separately in its bibliography.
:::

## Selecting connectivity in a chemical state

Both connectivity tools accept keyword-only `chemical_state` and
`structure_indices`. The reference remains the default; an integer chooses an
explicit state. With `chemical_state='structure'`, selected source structures
must resolve to one known state. Missing associations or multiple states raise;
there is no fallback to reference. Structure indices also constrain spatial
selection. General `get` propagates that same scope to selections and masks.

The graph is induced by the selected atom set: omitted atoms cannot provide
paths between retained atoms. Nodes and returned sets use full source atom
indices, with duplicates removed and blocks ordered by their smallest atom index.
The label array covers the full source atom axis and uses -1 outside selection.
Empty selections return an empty set array or an all-minus-one label array.
These operations inspect stored relationships; they do not infer missing bonds
or certify a partial graph complete. Existing relationship-type behavior remains.
Legacy 0.1 topology/system dictionaries remain readable as historical inputs.
The 0.2 TopologyDict stores stable inventory only; MolSysDict 0.2 and H5MSM 0.5
preserve the independent chemical collection, including additional states.

This independent six-atom chain illustrates why filtering full-system component
labels cannot replace induced connectivity:


In [7]:
import numpy as np
import pandas as pd
import molsysmt as msm
from molsysmt.native import MolSys
molsys = MolSys(n_atoms=6)
molsys.chemical_states._set_chemical_state_bonds(pd.DataFrame(dict(atom1_index=[0, 1, 2, 3, 4],
                               atom2_index=[1, 2, 3, 4, 5])))
selected = [5, 1, 0, 3, 1]
graph = msm.chemical_states.get_bondgraph(molsys, selection=selected, chemical_state=0)
assert list(graph.nodes) == [0, 1, 3, 5]
assert list(graph.edges) == [(0, 1)]
labels = msm.chemical_states.get_covalent_blocks(
    molsys, selection=selected, chemical_state=0, output_type='numpy.ndarray')
assert labels.tolist() == [0, 0, -1, 1, -1, 2]
print(labels.tolist())


[0, 0, -1, 1, -1, 2]


### 4. Mapping sequence letters to source groups

Identity is the percentage of identical aligned letters divided by the
number of **query letters**. Caps, waters, ions and other non-amino-acid groups
do not contribute letters when the input is a molecular inventory. The returned
lists contain the corresponding **original group indices** in each source;
they are neither compacted sequence positions nor group IDs. Unequal sequence
lengths can give different identity percentages when query and reference are
reversed. Selections with no letters, and supplied sequences with existing gap
characters, raise an explicit argument error.

The existing counting approach is adapted from Ensembler; MolSysMT retains
query-length normalization, whereas that reference uses the shorter sequence.
Biopython performs the global alignment. Optional Ackredit records the adapted
reference and the actual Biopython version after successful computation.

The [identity tutorial](../../user/tools/topology/get_sequence_identity.ipynb) provides the complete comparison workflow.


In [8]:
molsys_1 = msm.convert(msm.systems['pentalanine']['traj_pentalanine.h5msm'],
                       structure_indices=[0])
molsys_2 = msm.extract(molsys_1, selection="group_type=='amino acid'")
identity, query_groups, reference_groups = msm.topology.get_sequence_identity(
    molsys_1, reference_molecular_system=molsys_2)
assert identity == 100.0
assert query_groups == [1, 2, 3, 4, 5]
assert reference_groups == [0, 1, 2, 3, 4]
print('Identity:', identity)
print('Query source group indices:', query_groups)
print('Reference source group indices:', reference_groups)


Identity: 100.0
Query source group indices: [1, 2, 3, 4, 5]
Reference source group indices: [0, 1, 2, 3, 4]


### Selecting native sequence groups

Sequence recognition, group extraction and one/three-letter translation are
native MolSysMT operations. Biopython is optional for explicitly requested
Seq/SeqRecord outputs and the alignment algorithms that use it. Select original
three-letter groups before omitting terminal caps; repeated positions retain their
order. Empty object conversions are valid, while empty alignment/identity inputs
remain invalid. Representation prefixes never become biological sequence letters.


In [9]:
native_letters = msm.convert('AlaGly', to_form='string:amino_acids_1')
assert native_letters == 'AG'
assert msm.convert(native_letters, to_form='string:amino_acids_3') == 'AlaGly'
selected_sequence = msm.convert('amino_acids_3:AceAlaGlyNme',
    selection=[2, 1, 2, 0], to_form='biopython.Seq')
assert str(selected_sequence) == 'GAG'
assert str(msm.convert('amino_acids_3:AlaGly', selection=[], to_form='biopython.Seq')) == ''
print('Native alphabets and original sequence-group selections verified.')


Native alphabets and original sequence-group selections verified.


(course-core-covalent-connectivity-fasta-sources)=
## Inspecting submitted FASTA

Before converting a submitted sequence library, preserve its source records
with `msm.form.file_fasta.parse_records` or `read_records`. They do not require
Biopython or create topology/coordinates. A record index identifies one source
occurrence, independently of repeated headers/sequences; it is not a group
index. Only line terminators are removed from the joined sequence. Case,
spaces, leading M and symbols are retained for your own admission policy.
Existing SeqRecord conversions keep their normalized behavior. See
{ref}`FASTA ingestion <Tutorial_Form_file_fasta>`.


In [10]:
fasta_text = ">submitted description\n acD \nA*\n>empty\n"
fasta_source = msm.form.file_fasta.parse_records(fasta_text, source_label="submitted")
assert [record["sequence"] for record in fasta_source["records"]] == [" acD A*", ""]
assert fasta_source["preamble"] + "".join(
    record["raw_record"] for record in fasta_source["records"]) == fasta_text
print("Source record indices:", [r["record_index"] for r in fasta_source["records"]])


Source record indices: [0, 1]


:::{hint}
:class: dropdown
`msm.form.file_fasta.parse_records` inspects already decoded text;
`read_records` reads strict UTF-8 without newline translation. Both preserve
source occurrences before scientific admission. See
{func}`molsysmt.form.file_fasta.parse_records` and
{func}`molsysmt.form.file_fasta.read_records` for the result schema.
:::


(course-core-covalent-connectivity-group-inventories)=
### Native group inventories

A sequence and a set of sequences are partial molecular-system representations.
You can convert one-/three-letter strings to native Topology or MolSys without
constructing atoms. Three-letter caps remain groups. FASTA conversion provides
one declared molecule occurrence per record, including repeats; records are not
coordinate structures. The old implicit CA-per-position projection is removed.

You can use native `get()` or `msm.get()` to project groups and their chains,
molecules and entities without passing through atoms. Textual selectors query
that declared hierarchy. `msm.extract(..., element="group", selection=[2, 0])`
retains requested group order and remaps memberships; `element="chain"` retains
whole chains, including empty records. FASTA chain IDs are generated as `A`
through `Z`, then `AA`, `AB`, and onward; headers remain separate metadata.
The ordered chain/group relation survives TopologyDict and H5MSM 0.5 round trips.

Stored atom count zero means an unpopulated representation, not a physically
atom-free molecule. Atomic connectivity and geometry still require an explicit
realization. A single one-letter string cannot encode several chain boundaries.
See {ref}`Tutorial_Form_string_amino_acids_3`, {ref}`Tutorial_Form_file_fasta`
and {ref}`Cookbook_Sequence_source_groups`.


In [11]:
molsys = msm.convert("amino_acids_3:AceAlaGlyNme", to_form="molsysmt.MolSys")
assert molsys.topology.n_atoms == 0
assert molsys.structures.n_structures == 0
assert msm.get(molsys, element="group", group_name=True) == ["ACE", "ALA", "GLY", "NME"]
print("Represented groups:", molsys.topology.n_groups)
print("Stored atoms:", molsys.topology.n_atoms)


Represented groups: 4
Stored atoms: 0


--- 

### 🏆 Challenge 13: The Sequence & Topology Master

1. Load **T4 Lysozyme** (`pdb_id:181L`) and the variant structure (`pdb_id:5X33`).
2. Partition T4 Lysozyme into covalent blocks using `msm.chemical_states.get_covalent_blocks()`.
3. Compute the sequence alignment between both structures using `msm.topology.get_sequence_alignment()`.
4. Verify that their sequence identity percentage is around 52.65% using `msm.topology.get_sequence_identity()`.

Topological analysis bridges chemistry, graph theory, and bioinformatics. In {ref}`Module 14: System Comparison and Validation <course-core-comparing-systems>`, we will explore 3D structural alignment and RMSD calculations.

```{key-takeaway}
The `msm.chemical_states` module partitions covalent blocks and measures covalent paths in a selected chemical state. The `msm.topology` module aligns amino-acid sequences and calculates sequence identity from stable groups.
```

(course-core-covalent-connectivity-see-also)=
:::{seealso}
:class: dropdown
**API Documentation for Functions in this Module:**
- {func}`molsysmt.chemical_states.get_covalent_blocks` — Covalent block partition tool.
- {func}`molsysmt.topology.get_sequence_alignment` — Pairwise sequence alignment engine.
- {func}`molsysmt.topology.get_sequence_identity` — Sequence identity calculator.

**Related Course Modules & Guides:**
- Previous Module: [Module 12: Building, Repairing and Auditing Systems](../Common_Core/12_Building_Repairing_and_Auditing_Systems.ipynb)
- Next Module: {ref}`Module 14: System Comparison and Validation <course-core-comparing-systems>`
- User Guide: {ref}`user-foundations-index`
:::